<div style="text-align: center; line-height: 0; padding-top: 9px;">
  <img src="./images/btp-banner.gif" alt="BTP A&C">
</div>

## Enhanced Agentic RAG with SAP Generative AI Hub

In this advanced demo, we will explore how to build an **enhanced agentic RAG system** using SAP Generative AI Hub, LangGraph, and SAP HANA Cloud Vector Engine. This implementation builds upon traditional RAG by introducing autonomous agent capabilities that can intelligently select and use multiple tools, manage conversation history, and provide enterprise-grade features like error handling, observability, and result reranking.

The agent architecture enables the system to autonomously decide which data sources to query (supplier contracts, purchase orders, quality incidents, delivery reports) based on user questions, synthesize information from multiple sources, and provide comprehensive, evidence-based answers with proper citations and traceability.

## 🎯 Learning Objectives

By the end of this demo, you will be able to:
- Design and implement an autonomous agent system that intelligently selects and orchestrates multiple retrieval tools based on query analysis
- Build production-ready RAG systems with enterprise features including memory management, error handling, logging, and performance monitoring
- Implement result reranking strategies to improve answer quality across multiple data sources
- Create evaluation frameworks to measure and improve agent performance
- Configure and optimize agent behavior through centralized configuration management
- Understand the differences between streaming and non-streaming response modes for real-time user experiences

## 🚨 Requirements

Please review the following requirements before starting the demo:
- Complete the notebook **Retrieval-Augmented Generation with SAP HANA Cloud Vector Engine** (2-rag_with_hana_vectordb.ipynb)
- Complete the notebook **Agentic RAG with SAP Generative AI Hub** (5-agentic-rag.ipynb)
- Basic understanding of LangChain and LangGraph concepts
- Familiarity with agent-based systems and tool use patterns

### Step 1: Install Python packages

Run the following package installations. **pip** is the package installer for Python. You can use pip to install packages from the Python Package Index and other indexes.

⚠️ **Note:** Jupyter Notebook kernel restart required after package installation.

In [ ]:
%pip install -U langchain_core
%pip install -U gen-ai-hub-proxy-langchain
%pip install -U langgraph 
%pip install -U sap-ai-sdk-gen[all] 
%pip install -U hdbcli 
%pip install -U langchain-hana 
%pip install -U python-dotenv
%pip install -U sentence-transformers  # For reranking

# kernel restart required!!!

### Step 2: Import required libraries

Import all necessary Python modules for building the enhanced agentic RAG system, including LangChain components for agent orchestration, SAP AI Core SDK for model access, and supporting libraries for logging and data management.

In [ ]:
import os
import json
import logging
from typing import List, Dict, Any, Optional
from datetime import datetime
import urllib3
urllib3.disable_warnings()

# LangChain and LangGraph imports
from langchain_core.messages import SystemMessage, HumanMessage, AIMessage
from langchain.tools import tool
from langgraph.graph import START, StateGraph, MessagesState
from langgraph.prebuilt import tools_condition, ToolNode
from langgraph.checkpoint.memory import MemorySaver

# SAP AI Core imports
from ai_core_sdk.ai_core_v2_client import AICoreV2Client
from gen_ai_hub.proxy.gen_ai_hub_proxy import GenAIHubProxyClient
from gen_ai_hub.proxy.langchain import OpenAIEmbeddings, ChatOpenAI, ChatBedrock
from gen_ai_hub.proxy.langchain.init_models import init_llm

# HANA imports
from hdbcli import dbapi
from langchain_hana import HanaDB

# Environment variables
from dotenv import load_dotenv
load_dotenv()

print("✅ Libraries imported successfully!")

### Step 3: Configure centralized system settings

Establish a centralized configuration dictionary that controls all aspects of the agent system. This approach enables easy experimentation with different parameters (model selection, retrieval settings, agent behavior) without modifying the code throughout the notebook. Key configurations include:
- **Model Configuration**: LLM selection, token limits, and temperature settings
- **Retrieval Configuration**: Number of results to retrieve and reranking settings
- **Agent Configuration**: Iteration limits, streaming behavior, and memory management
- **Logging Configuration**: Verbosity level and token usage tracking

In [ ]:
# Centralized configuration
CONFIG = {
    # Model Configuration
    "llm_model": "gpt-5",
    "embedding_model": "text-embedding-3-large",
    "max_tokens": 4000,
    "temperature": 0.1,
    
    # Retrieval Configuration
    "top_k_results": 5,
    "rerank_top_k": 5,
    "enable_reranking": True,
    
    # Agent Configuration
    "max_iterations": 15,
    "enable_streaming": True,
    "enable_memory": True,
    
    # Logging Configuration
    "log_level": "INFO",
    "log_token_usage": True,
    
    # Table Names
    "tables": {
        "contracts": "SUPPLIER_CONTRACTS_DEFAULTPOWER",
        "purchase_orders": "PURCHASE_ORDERS_DEFAULTPOWER",
        "quality_incidents": "QUALITY_INCIDENTS_DEFAULTPOWER",
        "delivery_reports": "DELIVERY_REPORT_DEFAULTPOWER"
    }
}

# Setup logging
logging.basicConfig(
    level=getattr(logging, CONFIG["log_level"]),
    format='%(asctime)s - %(name)s - %(levelname)s - %(message)s'
)
logger = logging.getLogger(__name__)

print("✅ Configuration loaded successfully!")
print(json.dumps(CONFIG, indent=2))

### Step 4: Configure AI Core Client

Execute the configuration module below to enable access to SAP's Generative AI foundation models. Running this code block will automatically handle the necessary setup, including authentication and environment configuration, to ensure seamless connectivity to the Generative AI services.

In [ ]:
try:
    # Get the AI Core credentials from environment variables
    URL = os.getenv('AICORE_AUTH_URL')
    CLIENT_ID = os.getenv('AICORE_CLIENT_ID')
    CLIENT_SECRET = os.getenv('AICORE_CLIENT_SECRET')
    RESOURCE_GROUP = os.getenv('AICORE_RESOURCE_GROUP')
    API_URL = os.getenv('AICORE_BASE_URL')

    if not all([URL, CLIENT_ID, CLIENT_SECRET, RESOURCE_GROUP, API_URL]):
        raise ValueError("Missing required AI Core environment variables")

    # Set up the AICoreV2Client
    ai_core_client = AICoreV2Client(
        base_url=API_URL,
        auth_url=URL,
        client_id=CLIENT_ID,
        client_secret=CLIENT_SECRET,
        resource_group=RESOURCE_GROUP
    )

    # Initialize GenAIHub proxy client
    proxy_client = GenAIHubProxyClient(ai_core_client=ai_core_client)
    logger.info("AI Core client initialized successfully")
    print("✅ AI Core Client connection is established successfully!")
    
except Exception as e:
    logger.error(f"Failed to initialize AI Core client: {str(e)}")
    raise

Initialize the embedding and LLM models

In [ ]:
try:
    # Initialize embedding and LLM models
    embedding_model = OpenAIEmbeddings(
        proxy_model_name=CONFIG["embedding_model"],
        proxy_client=proxy_client
    )
    
    llm = init_llm(
        CONFIG["llm_model"],
        max_tokens=CONFIG["max_tokens"],
        temperature=CONFIG["temperature"]
    )
    
    logger.info(f"Models initialized: {CONFIG['llm_model']}, {CONFIG['embedding_model']}")
    print("✅ AI models are initialized successfully!")
    
except Exception as e:
    logger.error(f"Failed to initialize models: {str(e)}")
    raise

### Step 5: Connect to SAP HANA Cloud database

The provided Python script imports database connection modules and initiates a connection to a SAP HANA Cloud instance using the **dbapi** module. The connection credentials are retrieved from environment variables to ensure secure access to the SAP HANA Cloud database.

In [ ]:
try:
    # Get the HANA Cloud credentials from environment variables
    HANA_USER = os.getenv('HANA_VECTOR_USER')
    HANA_PASS = os.getenv('HANA_VECTOR_PASS')
    HANA_HOST = os.getenv('HANA_VECTOR_HOST')

    if not all([HANA_USER, HANA_PASS, HANA_HOST]):
        raise ValueError("Missing required HANA Cloud environment variables")

    # Establish connection to SAP HANA Cloud database
    conn = dbapi.connect(
        user=HANA_USER,
        password=HANA_PASS,
        address=HANA_HOST,
        port=443,
    )
    cursor = conn.cursor()
    
    logger.info("HANA Cloud connection established")
    print("✅ HANA Cloud connection is established successfully!")
    
except Exception as e:
    logger.error(f"Failed to connect to HANA Cloud: {str(e)}")
    raise

### Step 6: Implement result reranking

Result reranking is a crucial technique to improve answer quality when retrieving information from multiple sources. After initial vector similarity search retrieves candidate documents, a cross-encoder model re-scores these results based on their actual relevance to the query. This two-stage approach (fast retrieval + accurate reranking) significantly improves the precision of information provided to the LLM, leading to better final answers.

In [ ]:
from sentence_transformers import CrossEncoder

# Initialize reranker model (lightweight cross-encoder)
reranker_model = None

def initialize_reranker():
    """Initialize the reranker model lazily."""
    global reranker_model
    if reranker_model is None and CONFIG["enable_reranking"]:
        try:
            reranker_model = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')
            logger.info("Reranker model initialized")
        except Exception as e:
            logger.warning(f"Failed to initialize reranker: {str(e)}")
            reranker_model = None
    return reranker_model

def rerank_documents(query: str, documents: List[Any], top_k: int = None) -> List[Any]:
    """
    Rerank documents using a cross-encoder model.
    
    Args:
        query: The user query
        documents: List of document objects with page_content attribute
        top_k: Number of top documents to return (default from config)
    
    Returns:
        Reranked and filtered list of documents
    """
    if not documents:
        return documents
    
    if top_k is None:
        top_k = CONFIG["rerank_top_k"]
    
    if not CONFIG["enable_reranking"]:
        return documents[:top_k]
    
    try:
        reranker = initialize_reranker()
        if reranker is None:
            return documents[:top_k]
        
        # Prepare query-document pairs
        pairs = [[query, doc.page_content] for doc in documents]
        
        # Get scores
        scores = reranker.predict(pairs)
        
        # Sort documents by score
        scored_docs = list(zip(documents, scores))
        scored_docs.sort(key=lambda x: x[1], reverse=True)
        
        # Return top-k documents
        reranked_docs = [doc for doc, score in scored_docs[:top_k]]
        
        logger.info(f"Reranked {len(documents)} documents to top {top_k}")
        return reranked_docs
        
    except Exception as e:
        logger.warning(f"Reranking failed: {str(e)}. Returning original top-k.")
        return documents[:top_k]

print("✅ Reranker initialized!")

### Step 7: Define agent tools with enhanced capabilities

Define the specialized tools that the agent can autonomously select and use to retrieve information. Each tool is designed as an independent function with:
- **Clear documentation**: Describes when and how to use the tool
- **Error handling**: Graceful failure with informative error messages
- **Structured outputs**: JSON-formatted results with metadata for traceability
- **Logging**: Tracks tool usage for debugging and monitoring

The agent will intelligently choose which tools to invoke based on the user's question and the tool descriptions.

In [ ]:
def format_tool_results(documents: List[Any], tool_name: str) -> Dict[str, Any]:
    """
    Format tool results with metadata and citations.
    
    Args:
        documents: List of retrieved documents
        tool_name: Name of the tool that retrieved the documents
    
    Returns:
        Structured result dictionary
    """
    results = []
    for idx, doc in enumerate(documents):
        results.append({
            "rank": idx + 1,
            "content": doc.page_content,
            "metadata": doc.metadata,
            "source": tool_name
        })
    
    return {
        "tool": tool_name,
        "count": len(results),
        "results": results,
        "timestamp": datetime.now().isoformat()
    }

@tool(response_format="content_and_artifact")
def get_supplier_contract(name: str):
    """Returns the most relevant supplier contract data based on the supplier name.
    
    Use this tool when you need information about:
    - Contract terms and conditions
    - Late delivery penalties
    - Contract validity and end dates
    - Pricing agreements
    
    Args:
        name: The name of the supplier (e.g., 'Acme Corp', 'Sigma Electronics')
        
    Returns:
        Structured contract information with metadata
    """
    try:
        logger.info(f"Retrieving supplier contracts for: {name}")
        
        # Initialize HanaDB with existing table
        db = HanaDB(
            embedding=embedding_model,
            connection=conn,
            table_name=CONFIG["tables"]["contracts"]
        )

        # Perform similarity search
        contracts = db.similarity_search(name, k=CONFIG["top_k_results"])
        
        if not contracts:
            logger.warning(f"No contracts found for supplier: {name}")
            return "No supplier contract data found for the given name.", []
        
        # Format results
        formatted = format_tool_results(contracts, "get_supplier_contract")
        serialized = json.dumps(formatted, indent=2)
        
        logger.info(f"Retrieved {len(contracts)} contract documents")
        return serialized, contracts
        
    except Exception as e:
        logger.error(f"Error in get_supplier_contract: {str(e)}")
        return f"Error retrieving supplier contracts: {str(e)}", []

In [ ]:
@tool(response_format="content_and_artifact")
def get_purchase_order(name: str):
    """Returns the most relevant purchase order data based on the supplier name.
    
    Use this tool when you need information about:
    - Order status and fulfillment
    - Order quantities and dates
    - Payment terms
    
    Args:
        name: The name of the supplier
        
    Returns:
        Structured purchase order information
    """
    try:
        logger.info(f"Retrieving purchase orders for: {name}")
        
        db = HanaDB(
            embedding=embedding_model,
            connection=conn,
            table_name=CONFIG["tables"]["purchase_orders"]
        )

        purchase_orders = db.similarity_search(name, k=CONFIG["top_k_results"])
        
        if not purchase_orders:
            logger.warning(f"No purchase orders found for supplier: {name}")
            return "No purchase order data found for the given name.", []
        
        formatted = format_tool_results(purchase_orders, "get_purchase_order")
        serialized = json.dumps(formatted, indent=2)
        
        logger.info(f"Retrieved {len(purchase_orders)} purchase order documents")
        return serialized, purchase_orders
        
    except Exception as e:
        logger.error(f"Error in get_purchase_order: {str(e)}")
        return f"Error retrieving purchase orders: {str(e)}", []

In [ ]:
@tool(response_format="content_and_artifact")
def get_quality_incident(name: str):
    """Returns the most relevant quality incident data based on the supplier name.
    
    Use this tool when you need information about:
    - Product quality issues
    - Incident severity levels
    - Resolution status
    - Quality metrics and trends
    
    Args:
        name: The name of the supplier
        
    Returns:
        Structured quality incident information
    """
    try:
        logger.info(f"Retrieving quality incidents for: {name}")
        
        db = HanaDB(
            embedding=embedding_model,
            connection=conn,
            table_name=CONFIG["tables"]["quality_incidents"]
        )

        incidents = db.similarity_search(name, k=CONFIG["top_k_results"])
        
        if not incidents:
            logger.warning(f"No quality incidents found for supplier: {name}")
            return "No quality incident data found for the given name.", []
        
        formatted = format_tool_results(incidents, "get_quality_incident")
        serialized = json.dumps(formatted, indent=2)
        
        logger.info(f"Retrieved {len(incidents)} quality incident documents")
        return serialized, incidents
        
    except Exception as e:
        logger.error(f"Error in get_quality_incident: {str(e)}")
        return f"Error retrieving quality incidents: {str(e)}", []

In [ ]:
@tool(response_format="content_and_artifact")
def get_delivery_report(name: str):
    """Returns the most relevant delivery report data based on the supplier name.
    
    Use this tool when you need information about:
    - Delivery timeliness (delays, on-time, early)
    - Delivery performance
    - Shipment details
    
    Args:
        name: The name of the supplier
        
    Returns:
        Structured delivery report information
    """
    try:
        logger.info(f"Retrieving delivery reports for: {name}")
        
        db = HanaDB(
            embedding=embedding_model,
            connection=conn,
            table_name=CONFIG["tables"]["delivery_reports"]
        )

        reports = db.similarity_search(name, k=CONFIG["top_k_results"])
        
        if not reports:
            logger.warning(f"No delivery reports found for supplier: {name}")
            return "No delivery report data found for the given name.", []
        
        formatted = format_tool_results(reports, "get_delivery_report")
        serialized = json.dumps(formatted, indent=2)
        
        logger.info(f"Retrieved {len(reports)} delivery report documents")
        return serialized, reports
        
    except Exception as e:
        logger.error(f"Error in get_delivery_report: {str(e)}")
        return f"Error retrieving delivery reports: {str(e)}", []

In [ ]:
# Collect all tools
tools = [
    get_supplier_contract,
    get_purchase_order,
    get_quality_incident,
    get_delivery_report
]

print(f"✅ {len(tools)} tools defined successfully!")

### Step 8: Implement query analysis for intelligent tool routing

The query analyzer examines the user's question to determine which tools are most relevant before the agent begins execution. This optimization:
- Reduces unnecessary tool calls by identifying intent upfront
- Provides insights into the query type for better handling
- Helps the agent make more informed decisions about tool selection

For example, a query about "delivery performance" will suggest the delivery_report tool, while a question about "contract penalties" will route to the contract tool.

In [ ]:
def analyze_query(query: str) -> Dict[str, Any]:
    """
    Analyze the user query to extract key information and suggest relevant tools.
    
    Args:
        query: The user's question
    
    Returns:
        Dictionary with analysis results
    """
    query_lower = query.lower()
    
    analysis = {
        "query": query,
        "suggested_tools": [],
        "keywords": [],
        "query_type": "general"
    }
    
    # Detect query intent - check for "overall performance" first
    if 'performance' in query_lower and 'overall' in query_lower:
        analysis["suggested_tools"] = [tool.name for tool in tools]
        analysis["query_type"] = "comprehensive"
        return analysis
    
    if any(word in query_lower for word in ['contract', 'penalty', 'agreement', 'term']):
        analysis["suggested_tools"].append("get_supplier_contract")
        analysis["query_type"] = "contract"
    
    if any(word in query_lower for word in ['order', 'purchase', 'po']):
        analysis["suggested_tools"].append("get_purchase_order")
        analysis["query_type"] = "purchase_order"
    
    if any(word in query_lower for word in ['quality', 'defect', 'incident', 'issue']):
        analysis["suggested_tools"].append("get_quality_incident")
        analysis["query_type"] = "quality"
    
    if any(word in query_lower for word in ['delivery', 'delay', 'late', 'on-time', 'early']):
        analysis["suggested_tools"].append("get_delivery_report")
        analysis["query_type"] = "delivery"
    
    # If no specific tool identified, suggest all for comprehensive search
    if not analysis["suggested_tools"]:
        analysis["suggested_tools"] = [tool.name for tool in tools]
        analysis["query_type"] = "comprehensive"
    
    logger.info(f"Query analysis: type={analysis['query_type']}, tools={analysis['suggested_tools']}")
    return analysis

print("✅ Query analyzer ready!")

### Step 9: Configure system prompt for the agent

The system message provides crucial instructions to the LLM about how to behave as an analyst. A well-crafted system prompt ensures the agent:
- Understands its role and responsibilities
- Follows consistent response formatting
- Properly cites sources and shows calculations
- Handles missing data appropriately
- Synthesizes information from tool results correctly

In [ ]:
sys_msg = SystemMessage(content="""
You are an AI analyst for Supplier Performance Management.

TOOL USAGE INSTRUCTIONS:
1. First, call the necessary tools to gather data
2. After receiving tool results, ANALYZE the JSON data returned
3. Extract relevant information from the tool results
4. Formulate a comprehensive answer based on the data

RESPONSE REQUIREMENTS (AFTER TOOLS):
- Start with a direct answer to the user's question
- Include specific numbers, dates, and metrics from the retrieved data
- Cite sources: (Source: [Tool Name] | Supplier: [Name])
- Show calculations if computing metrics
- Never fabricate information not in the tool results

IMPORTANT: After calling all necessary tools, you MUST provide a final answer. Do not leave the response empty.

Example Response Format:
"Based on the retrieved data, Supplier X shows the following:

Delivery Performance:
- 15 deliveries tracked
- On-time rate: 80% (12 out of 15 deliveries)
(Source: get_delivery_report | Supplier: X)

Overall Assessment: [Your assessment here]"

Domain Rules:
- Delay values: negative=early, 0=on-time, positive=late
- Always extract and use specific data points from tool results
""")

print("✅ System message configured!")

### Step 10: Build the agent graph with memory

Create the agent's computational graph using LangGraph, which defines how the agent processes user inputs and decides when to use tools. The graph includes:
- **Assistant Node**: Makes decisions about which tools to call and generates final responses
- **Tools Node**: Executes the selected tools and returns results
- **Memory/Checkpointing**: Enables conversation history tracking across multiple turns
- **Conditional Edges**: Routes between assistant and tools based on whether tool calls are needed

In [ ]:
# Bind tools to LLM
llm_with_tools = llm.bind_tools(tools)

print("✅ LLM with tools bound successfully!")

In [ ]:
def assistant(state: MessagesState):
    """Assistant node with enhanced logging."""
    try:
        response = llm_with_tools.invoke([sys_msg] + state["messages"])
        
        # Log token usage if available
        if CONFIG["log_token_usage"] and hasattr(response, 'response_metadata'):
            usage = response.response_metadata.get('token_usage', {})
            logger.info(f"Token usage: {usage}")
        
        return {"messages": [response]}
        
    except Exception as e:
        logger.error(f"Error in assistant node: {str(e)}")
        error_msg = AIMessage(content=f"I encountered an error: {str(e)}")
        return {"messages": [error_msg]}

In [ ]:
# Build the graph
builder = StateGraph(MessagesState)

# Add nodes
builder.add_node("assistant", assistant)
builder.add_node("tools", ToolNode(tools))

# Add edges
builder.add_edge(START, "assistant")
builder.add_conditional_edges(
    "assistant",
    tools_condition,
)
builder.add_edge("tools", "assistant")

# Add memory/checkpointing
if CONFIG["enable_memory"]:
    memory = MemorySaver()
    graph = builder.compile(checkpointer=memory)
    logger.info("Graph compiled with memory checkpointing")
else:
    graph = builder.compile(recursion_limit=20)
    logger.info("Graph compiled without memory")

print("✅ Agent graph built successfully!")

Visualize the agent graph structure

In [ ]:
# Visualize the graph
try:
    from IPython.display import Image, display
    display(Image(graph.get_graph().draw_mermaid_png()))
except Exception as e:
    print(f"Could not visualize graph: {str(e)}")

### Step 11: Execute queries with the agent

The execute_query function provides a unified interface for interacting with the agent. It supports two modes:
- **Streaming Mode** (default): Displays responses progressively as they're generated, similar to ChatGPT's typing effect
- **Non-Streaming Mode**: Waits for the complete response before displaying, useful for debugging

The function also performs query analysis before execution and manages conversation threading for multi-turn interactions.

In [ ]:
def execute_query(user_input: str, thread_id: str = "default", stream: bool = None):
    """
    Execute a query with the agent.
    
    Args:
        user_input: The user's question
        thread_id: Conversation thread ID for memory
        stream: Whether to stream the response (default from config)
    
    Returns:
        Final response message
    """
    if stream is None:
        stream = CONFIG["enable_streaming"]
    
    # Analyze query
    analysis = analyze_query(user_input)
    print(f"\n📊 Query Analysis: {analysis['query_type'].upper()}")
    print(f"🔧 Suggested tools: {', '.join(analysis['suggested_tools'])}\n")
    
    # Prepare config for conversation threading
    config = {"configurable": {"thread_id": thread_id}}
    
    if stream:
        print("💬 Streaming response...\n")
        print("="*80)
        
        # Stream the response
        for event in graph.stream(
            {"messages": [("user", user_input)]},
            config=config,
            stream_mode="values"
        ):
            message = event["messages"][-1]
            if hasattr(message, 'content') and message.content:
                message.pretty_print()
        
        print("="*80)
        return event["messages"][-1]
    else:
        # Non-streaming execution
        result = graph.invoke(
            {"messages": [("user", user_input)]},
            config=config
        )
        
        print("\n💬 Response:\n")
        print("="*80)
        for message in result["messages"]:
            message.pretty_print()
        print("="*80)
        
        return result["messages"][-1]

print("✅ Query execution helper ready!")

### Step 12: Test the enhanced agent

Run example queries to test the agent's capabilities. Each query demonstrates different aspects:
- Single-tool queries (e.g., contract information)
- Multi-tool queries (e.g., overall performance requiring multiple data sources)
- Conversation memory (follow-up questions in the same thread)

In [ ]:
# Test Query 1: Delivery Performance
response = execute_query(
    "Tell me about PulseWave Materials delivery performance",
    thread_id="test_session_1",
    stream=False  # Use False to see all messages for debugging
)

In [ ]:
# Test Query 2: Contract Information
response = execute_query(
    "What is the late delivery penalty for Sigma Electronics Co.?",
    thread_id="test_session_2"
)

In [ ]:
# Test Query 3: Quality Issues
response = execute_query(
    "Show me quality incidents with high severity from PulseWave Materials Inc.",
    thread_id="test_session_3"
)

### Cleanup

Close database connections when done to free up resources.

In [ ]:
# Close HANA connection
try:
    if 'conn' in globals():
        conn.close()
        print("✅ HANA connection closed")
except Exception as e:
    print(f"Error closing connection: {str(e)}")